<h1> Assignment #2 EDA

<H2> Preliminary items

<H3> Loading and sampling

In [2]:
# Import libraries, data, and sampling

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("backend")
FULL_CSV = DATA_DIR / "miami-housing.csv"
COUNTY_GEOJSON = DATA_DIR / "miami_dade.geojson"
OUT_CSV = DATA_DIR / "miami_sample_25pct.csv"

SEED = 81
SAMPLE_FRAC = 0.25
SALE_YEAR = 2016   # Kaggle says these are 2016 sales -- verify before relying on it
N_BUCKETS = 5      # price buckets shared by every view
SHOW_PLOTS = False # True opens a window per figure; the script waits until you CLOSE it

In [3]:
# Loading and sampling

miami_full = pd.read_csv(FULL_CSV, dtype={"PARCELNO": str})
miami = miami_full.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
print(f"Full: {len(miami_full):,} rows | Sampled: {len(miami):,} rows")

Full: 13,932 rows | Sampled: 3,483 rows


<H3> Nullity checks, data loading, etc.

In [4]:
print("\nMissing values per column (only columns with any):")
missing = miami_full.isna().sum()
print(missing[missing > 0] if missing.any() else "  none")

print("\nDuplicate PARCELNO in full data:", miami_full["PARCELNO"].duplicated().sum())
print("Fully duplicated rows:          ", miami_full.duplicated().sum())

print("\nCoordinate ranges (sample):")
print(f"  latitude  {miami['LATITUDE'].min():.4f} to {miami['LATITUDE'].max():.4f}")
print(f"  longitude {miami['LONGITUDE'].min():.4f} to {miami['LONGITUDE'].max():.4f}")

print("\nKey columns (sample):")
print(miami[["SALE_PRC", "TOT_LVG_AREA", "SPEC_FEAT_VAL",
             "WATER_DIST", "OCEAN_DIST", "age"]].describe().round(1).T)

n_zero_feat = (miami["SPEC_FEAT_VAL"] == 0).sum()
print(f"\nHouses with $0 special features: {n_zero_feat:,} "
      f"({n_zero_feat / len(miami):.0%} of sample) -> size of the 'No special features' strip")


Missing values per column (only columns with any):
  none

Duplicate PARCELNO in full data: 156
Fully duplicated rows:           0

Coordinate ranges (sample):
  latitude  25.4388 to 25.9744
  longitude -80.5239 to -80.1197

Key columns (sample):
                count      mean       std      min       25%       50%  \
SALE_PRC       3483.0  401038.6  320692.1  72000.0  235000.0  310000.0   
TOT_LVG_AREA   3483.0    2062.9     815.4    858.0    1481.0    1887.0   
SPEC_FEAT_VAL  3483.0    9477.5   13791.7      0.0     737.0    2729.0   
WATER_DIST     3483.0   11869.3   11811.4      0.0    2744.1    6814.9   
OCEAN_DIST     3483.0   31608.7   17607.5    435.7   18258.9   28201.9   
age            3483.0      30.6      21.1      0.0      14.0      26.0   

                    75%        max  
SALE_PRC       426000.0  2637500.0  
TOT_LVG_AREA     2455.0     6016.0  
SPEC_FEAT_VAL   12225.0   110238.0  
WATER_DIST      18912.4    49840.9  
OCEAN_DIST      44043.4    75744.9  
age        

<H2> Advanced explorations

<h3> Derived columns

In [10]:
# Create new columns

miami["price_per_sqft"] = miami["SALE_PRC"] / miami["TOT_LVG_AREA"]
miami["feat_share_pct"] = 100 * miami["SPEC_FEAT_VAL"] / miami["SALE_PRC"]
miami["year_built"] = SALE_YEAR - miami["age"]
miami["decade_built"] = (miami["year_built"] // 10 * 10).astype(int)

# Price buckets
miami["price_bucket"], bucket_edges = pd.qcut(
    miami["SALE_PRC"], q=N_BUCKETS, labels=False, retbins=True
)
bucket_labels = [
    f"${bucket_edges[i] / 1e3:,.0f}k - ${bucket_edges[i + 1] / 1e3:,.0f}k"
    for i in range(N_BUCKETS)
]
print("\nPrice buckets:")
bucket_counts = miami["price_bucket"].value_counts().sort_index()
for b, count in bucket_counts.items():
    print(f"  {bucket_labels[b]:>16}: {count:,} houses")

# Remove dollar signes
plot_labels = [label.replace("$", r"\$") for label in bucket_labels]
dollar_axis = plt.FuncFormatter(lambda v, _: f"${v / 1e3:,.0f}k".replace("$", r"\$"))

# Coloring for buckets
bucket_colors = [plt.cm.viridis(x) for x in np.linspace(0.1, 0.9, N_BUCKETS)]
print("\nHex colors for D3:", [plt.matplotlib.colors.to_hex(c) for c in bucket_colors])

# Counting structure quality for categorization
miami["structure_quality"].value_counts().sort_index()
miami.groupby("structure_quality")["SALE_PRC"].agg(["count", "median"])


Price buckets:
      $72k - $220k: 735 houses
     $220k - $280k: 670 houses
     $280k - $342k: 686 houses
     $342k - $482k: 696 houses
   $482k - $2,638k: 696 houses

Hex colors for D3: ['#482475', '#355f8d', '#21918c', '#44bf70', '#bddf26']


,count,median
structure_quality,,
1,45,130000.0
2,1046,240000.0
3,2,1293000.0
4,1877,333000.0
5,513,591000.0


<H3> County map/GEOJSON test

In [6]:
def draw_county_outline(ax, path):
    """Draw a GeoJSON Polygon/MultiPolygon outline. Returns False if no file."""
    if not path.exists():
        return False
    geo = json.loads(path.read_text())
    features = geo["features"] if geo.get("type") == "FeatureCollection" else [geo]
    for feat in features:
        geom = feat.get("geometry", feat)
        polygons = (geom["coordinates"] if geom["type"] == "MultiPolygon"
                    else [geom["coordinates"]])
        for polygon in polygons:
            outer_ring = np.array(polygon[0])
            ax.fill(outer_ring[:, 0], outer_ring[:, 1],
                    facecolor="#f2efe8", edgecolor="#555", linewidth=1.2, zorder=0)
    return True


fig, ax = plt.subplots(figsize=(8, 9))
has_outline = draw_county_outline(ax, COUNTY_GEOJSON)

for b in range(N_BUCKETS):
    pts = miami[miami["price_bucket"] == b]
    ax.scatter(pts["LONGITUDE"], pts["LATITUDE"], # x = longitude; y = latitude
               s=6, color=bucket_colors[b], linewidths=0, alpha=0.8,
               label=plot_labels[b], zorder=2)

ax.set_aspect(1 / np.cos(np.radians(miami["LATITUDE"].mean())))
ax.set_title(f"Miami-Dade houses ({len(miami):,}-house sample, seed {SEED})"
             + ("" if has_outline else "\n(no county GeoJSON yet - houses only)"))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Sale price", loc="upper left", markerscale=3, frameon=True)
plt.tight_layout()
plt.savefig("prelim_map.png", dpi=150)
plt.show() if SHOW_PLOTS else plt.close(fig)
print("Saved prelim_map.png")

Saved prelim_map.png


<H3> Exploring view data

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
colors = [bucket_colors[b] for b in miami["price_bucket"]]

# View 1: water: distance to water vs. price
ax = axes[0]
ax.scatter(miami["WATER_DIST"], miami["SALE_PRC"], s=5, c=colors, linewidths=0, alpha=0.6)
ax.set_yscale("log")
ax.set_xlabel("Distance to water (ft)")
ax.set_ylabel("Sale price (log scale)")
ax.yaxis.set_major_formatter(dollar_axis)
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_title("View 1: proximity to water")

# View 2: building age (houses by decade built chronologically)
ax = axes[1]
decade_counts = miami["decade_built"].value_counts().sort_index()
ax.bar(decade_counts.index.astype(str) + "s", decade_counts.values, color="#7a8fa6")
ax.set_xlabel("Decade built")
ax.set_ylabel("Number of houses")
ax.set_title("View 2: building age")
ax.tick_params(axis="x", rotation=45)

# View 3: special features value addition (% of price vs. price; size = living area)
ax = axes[2]
has_feat = (miami["SPEC_FEAT_VAL"] > 0).to_numpy()
sizes = 4 + 60 * miami["TOT_LVG_AREA"] / miami["TOT_LVG_AREA"].max()
ax.scatter(miami.loc[has_feat, "feat_share_pct"], miami.loc[has_feat, "SALE_PRC"],
           s=sizes[has_feat], c=np.array(colors)[has_feat],
           linewidths=0, alpha=0.5)
ax.set_yscale("log")
ax.set_xlabel("Special features as % of sale price")
ax.set_ylabel("Sale price (log scale)")
ax.yaxis.set_major_formatter(dollar_axis)
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_title(f"View 3: special features ({n_zero_feat:,} houses at \\$0 not shown)")

plt.tight_layout()
plt.savefig("prelim_views.png", dpi=150)
plt.show() if SHOW_PLOTS else plt.close(fig)
print("Saved prelim_views.png")

Saved prelim_views.png


<H3> Miscellaneous

In [8]:
# Keep for backend just in case
keep = [
    "PARCELNO", "LATITUDE", "LONGITUDE", "SALE_PRC", "TOT_LVG_AREA",
    "SPEC_FEAT_VAL", "WATER_DIST", "OCEAN_DIST", "age",
    "price_per_sqft", "feat_share_pct", "year_built", "decade_built", "price_bucket",
]
miami[keep].to_csv(OUT_CSV, index=False)
print(f"Saved {len(miami):,} rows x {len(keep)} columns to {OUT_CSV}")

Saved 3,483 rows x 14 columns to backend/miami_sample_25pct.csv
